# Fuel Efficiency Modeling

This notebook predicts **combined fuel efficiency (`combined_mpg_ft1`)** from EPA-style vehicle attributes while emphasizing **reproducibility, leakage control, and interpretable evaluation**.

## What this notebook does
- locates a local CSV/ZIP dataset (or uses a Google Colab upload fallback)
- engineers a small set of row-level features such as **vehicle age** and parsed **engine descriptor** flags
- explicitly excludes target-derived and post-outcome fields from modeling
- compares a **mean baseline** against tree-based regressors with **MAE, RMSE, and R²**
- uses **grouped validation by vehicle family** when possible to reduce overly optimistic scores from repeated make/model variants
- saves compact artifacts to `artifacts/` only when data is available locally

> **Important:** This repository intentionally does **not** commit the external dataset. To populate the results table and figures with real values, place the EPA-style CSV/ZIP in `data/` (recommended), the notebook directory, or upload it in Colab, then run the notebook top to bottom.


In [ ]:
from pathlib import Path

RANDOM_STATE = 42
TEST_SIZE = 0.20
N_SPLITS = 5
N_JOBS = -1
DO_SHAP = False
SHAP_SAMPLE_N = 500
TARGET = None
TARGET_CANDIDATES = [
    'combined_mpg_ft1',
    'unrounded_combined_mpg_ft1',
    'combined_mpg_ft2',
    'combined_mpg',
    'mpg',
]

DATA_SEARCH_DIRS = [Path('data'), Path('.'), Path('/content')]
OUTDIR = Path('artifacts')
FIGDIR = OUTDIR / 'figures'


## Validation design

The EPA-style vehicle table contains repeated families of cars across trims and years. A purely random split can leak similarity between near-duplicate variants, which makes the score look better than the model's ability to generalize to unseen vehicle families.

**Default choice in this notebook:**
- Use **`GroupKFold` / `GroupShuffleSplit` on `make + model`** when both columns are present and there are enough groups.
- Fall back to shuffled random splits only when grouped validation is not possible.

This is usually more credible than a random split for this dataset. A time-aware holdout by model year is still useful when the goal is *future-year* forecasting, and the error-analysis section below highlights that limitation explicitly.


In [ ]:
import inspect
import json
import os
import random
import re
import warnings
import zipfile

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, make_scorer
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, KFold, ShuffleSplit, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

try:
    from IPython.display import Markdown, display
except Exception:
    def display(obj):
        print(obj)
    Markdown = str

try:
    from lightgbm import LGBMRegressor
except Exception:
    LGBMRegressor = None

try:
    from xgboost import XGBRegressor
except Exception:
    XGBRegressor = None

try:
    import shap
except Exception:
    shap = None

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 120)
warnings.filterwarnings('default')


def seed_everything(seed=RANDOM_STATE):
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)


seed_everything()

availability = pd.DataFrame(
    {
        'package': ['lightgbm', 'xgboost', 'shap'],
        'available': [LGBMRegressor is not None, XGBRegressor is not None, shap is not None],
    }
)
display(availability)


In [ ]:
def find_dataset_file(search_dirs):
    candidates = []
    for directory in search_dirs:
        directory = Path(directory)
        if not directory.exists():
            continue
        for pattern in ('*.csv', '*.zip'):
            for path in sorted(directory.glob(pattern)):
                if 'artifacts' in path.parts:
                    continue
                candidates.append(path)
    return candidates[0] if candidates else None


def load_vehicle_data(path):
    path = Path(path)
    if path.suffix.lower() == '.csv':
        return pd.read_csv(path, low_memory=False)
    if path.suffix.lower() == '.zip':
        with zipfile.ZipFile(path) as zf:
            csv_members = [name for name in zf.namelist() if name.lower().endswith('.csv')]
            if not csv_members:
                raise FileNotFoundError(f'No CSV found inside {path}')
            with zf.open(csv_members[0]) as handle:
                return pd.read_csv(handle, low_memory=False)
    raise ValueError(f'Unsupported dataset type: {path.suffix}')


def make_unique_columns(columns):
    seen = {}
    unique = []
    for col in columns:
        count = seen.get(col, 0)
        if count == 0:
            unique.append(col)
        else:
            unique.append(f'{col}__dup{count}')
        seen[col] = count + 1
    return unique


def coerce_numeric_like_cols(df, threshold=0.95, sample_size=500):
    df = df.copy()
    for col in df.columns:
        if not pd.api.types.is_object_dtype(df[col]):
            continue
        sample = df[col].dropna().astype(str).head(sample_size).str.replace(',', '', regex=False).str.strip()
        if sample.empty:
            continue
        parsable = sample.str.fullmatch(r'-?\d+(?:\.\d+)?')
        if parsable.mean() >= threshold:
            df[col] = pd.to_numeric(
                df[col].astype(str).str.replace(',', '', regex=False).str.strip(),
                errors='coerce',
            )
    return df


def first_present(df, candidates):
    for col in candidates:
        if col in df.columns:
            return col
    return None


def detect_target(df, candidates=TARGET_CANDIDATES):
    return first_present(df, candidates)


def parse_engine_descriptor(text):
    values = {
        'engine_displacement_from_text': np.nan,
        'cylinders_from_text': np.nan,
        'engine_has_turbo': 0,
        'engine_is_electrified': 0,
    }
    if pd.isna(text):
        return values
    text = str(text).lower()
    liters = re.search(r'(\d+(?:\.\d+)?)\s*[lL]\b', text)
    cylinders = re.search(r'(\d{1,2})\s*(?:cyl|cylinder)', text)
    if cylinders is None:
        cylinders = re.search(r'\b[iv](\d{1,2})\b', text)
    if liters:
        values['engine_displacement_from_text'] = float(liters.group(1))
    if cylinders:
        values['cylinders_from_text'] = float(cylinders.group(1))
    values['engine_has_turbo'] = int('turbo' in text)
    values['engine_is_electrified'] = int(any(token in text for token in ['hybrid', 'electric', 'ev', 'plug-in', 'phev', 'battery']))
    return values


def engineer_features(df):
    df = df.copy()
    year_col = first_present(df, ['model_year', 'year'])
    if year_col is not None:
        year_num = pd.to_numeric(df[year_col], errors='coerce')
        if year_num.notna().any():
            reference_year = int(year_num.max()) + 1
            df['vehicle_age'] = reference_year - year_num

    descriptor_col = first_present(df, ['engine_descriptor', 'eng_dscr', 'engine_description'])
    if descriptor_col is not None:
        parsed = df[descriptor_col].apply(parse_engine_descriptor).apply(pd.Series)
        df = pd.concat([df, parsed], axis=1)

    if 'displ' in df.columns:
        df['engine_displacement_l'] = pd.to_numeric(df['displ'], errors='coerce')
        if 'engine_displacement_from_text' in df.columns:
            df['engine_displacement_l'] = df['engine_displacement_l'].fillna(df['engine_displacement_from_text'])
    elif 'engine_displacement_from_text' in df.columns:
        df['engine_displacement_l'] = df['engine_displacement_from_text']

    if 'cylinders' in df.columns:
        df['cylinders_clean'] = pd.to_numeric(df['cylinders'], errors='coerce')
        if 'cylinders_from_text' in df.columns:
            df['cylinders_clean'] = df['cylinders_clean'].fillna(df['cylinders_from_text'])
    elif 'cylinders_from_text' in df.columns:
        df['cylinders_clean'] = df['cylinders_from_text']

    transmission_col = first_present(df, ['transmission', 'trany', 'transmission_desc'])
    if transmission_col is not None:
        df['transmission_gears_parsed'] = pd.to_numeric(
            df[transmission_col].astype(str).str.extract(r'(\d{1,2})')[0],
            errors='coerce',
        )

    fuel_col = first_present(df, ['fuel_type', 'fuel_type1', 'fuelType1'])
    if fuel_col is not None:
        fuel_text = df[fuel_col].fillna('').astype(str).str.lower()
        df['is_alt_powertrain'] = fuel_text.str.contains('electric|hybrid|cng|hydrogen|ethanol|flex').astype(int)

    return df


def build_exclusion_table(df, target):
    reasons = {}
    regex_rules = [
        (r'mpg', 'fuel-economy measurement / target-derived field'),
        (r'barrels|co2|ghg|fuel_cost|you_save_spend|range|charge|electricity', 'post-outcome emissions, cost, or range field'),
        (r'^id$|vehicle_id', 'identifier field'),
        (r'url|link', 'metadata field'),
    ]

    for col in df.columns:
        lower = col.lower()
        if col == target:
            reasons[col] = 'target column'
            continue
        for pattern, reason in regex_rules:
            if re.search(pattern, lower):
                reasons[col] = reason
                break

    for col in df.columns:
        if col in reasons:
            continue
        series = df[col]
        if series.notna().sum() == 0:
            reasons[col] = 'all values missing'
        elif series.nunique(dropna=True) <= 1:
            reasons[col] = 'constant / single-valued field'

    audit_rows = []
    y_num = pd.to_numeric(df[target], errors='coerce')
    for col in df.columns:
        if col == target:
            continue
        x_num = pd.to_numeric(df[col], errors='coerce')
        mask = x_num.notna() & y_num.notna()
        exact = np.nan
        near = np.nan
        corr = np.nan
        usable = int(mask.sum())
        if usable >= 50:
            exact = float((x_num[mask] == y_num[mask]).mean())
            near = float(np.isclose(x_num[mask], y_num[mask], rtol=0.01, atol=0.25).mean())
            if x_num[mask].nunique() > 1 and y_num[mask].nunique() > 1:
                corr = float(abs(x_num[mask].corr(y_num[mask])))
            if exact >= 0.98 or near >= 0.995 or (pd.notna(corr) and corr >= 0.995):
                reasons.setdefault(
                    col,
                    f'target-like numeric field (exact={exact:.3f}, near={near:.3f}, corr={corr:.3f})',
                )
        audit_rows.append(
            {
                'column': col,
                'numeric_overlap_rows': usable,
                'pct_exact_equal_to_target': exact,
                'pct_near_equal_to_target': near,
                'abs_correlation_with_target': corr,
                'excluded': col in reasons,
            }
        )

    exclusion_table = (
        pd.DataFrame({'column': list(reasons.keys()), 'reason': list(reasons.values())})
        .sort_values(['reason', 'column'])
        .reset_index(drop=True)
    )
    leakage_audit = pd.DataFrame(audit_rows).sort_values(
        ['excluded', 'pct_near_equal_to_target', 'abs_correlation_with_target'],
        ascending=[False, False, False],
        na_position='last',
    )
    return exclusion_table, leakage_audit


def build_groups(df):
    group_cols = [col for col in ['make', 'model'] if col in df.columns]
    if not group_cols:
        return None, []
    groups = df[group_cols].fillna('UNKNOWN').astype(str).agg(' | '.join, axis=1)
    return groups, group_cols


def choose_validation_strategy(groups, group_cols):
    if groups is not None and groups.nunique() >= N_SPLITS:
        return (
            GroupKFold(n_splits=N_SPLITS),
            GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE),
            f"Grouped validation by {' + '.join(group_cols)} to reduce family-level leakage across trims and years.",
        )
    return (
        KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE),
        ShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE),
        'Grouped columns were unavailable or too sparse, so the notebook fell back to a shuffled random split.',
    )


def build_preprocessor(X_frame):
    numeric_features = X_frame.select_dtypes(include=[np.number, 'bool']).columns.tolist()
    categorical_features = [col for col in X_frame.columns if col not in numeric_features]

    ohe_params = {}
    ohe_signature = inspect.signature(OneHotEncoder).parameters
    if 'min_frequency' in ohe_signature:
        ohe_params['min_frequency'] = 20
        ohe_params['handle_unknown'] = 'infrequent_if_exist'
    else:
        ohe_params['handle_unknown'] = 'ignore'
    if 'sparse_output' in ohe_signature:
        ohe_params['sparse_output'] = True
    else:
        ohe_params['sparse'] = True

    transformers = []
    if numeric_features:
        transformers.append(
            (
                'num',
                Pipeline([('imputer', SimpleImputer(strategy='median'))]),
                numeric_features,
            )
        )
    if categorical_features:
        transformers.append(
            (
                'cat',
                Pipeline([
                    ('imputer', SimpleImputer(strategy='most_frequent')),
                    ('encoder', OneHotEncoder(**ohe_params)),
                ]),
                categorical_features,
            )
        )

    preprocessor = ColumnTransformer(transformers=transformers, remainder='drop', sparse_threshold=1.0)
    return preprocessor, numeric_features, categorical_features


def rmse_score(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def save_json(payload, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as handle:
        json.dump(payload, handle, indent=2)


In [ ]:
DATA_PATH = find_dataset_file(DATA_SEARCH_DIRS)
if DATA_PATH is None:
    try:
        from google.colab import files
        uploaded = files.upload()
        if not uploaded:
            raise FileNotFoundError('No files were uploaded.')
        DATA_PATH = Path(next(iter(uploaded.keys())))
    except Exception as exc:
        raise FileNotFoundError(
            'No dataset was found. Place the EPA-style CSV/ZIP in data/, the notebook directory, or upload it in Colab.'
        ) from exc

OUTDIR.mkdir(exist_ok=True)
FIGDIR.mkdir(parents=True, exist_ok=True)

df_raw = load_vehicle_data(DATA_PATH)
df_raw.columns = make_unique_columns(df_raw.columns)
df = engineer_features(coerce_numeric_like_cols(df_raw))

display(pd.DataFrame({'rows': [len(df)], 'columns': [df.shape[1]], 'dataset_path': [str(DATA_PATH)]}))
display(df.head())


In [ ]:
target = TARGET or detect_target(df)
if target is None:
    raise ValueError(f'Could not detect a target column from: {TARGET_CANDIDATES}')

exclusion_table, leakage_audit = build_exclusion_table(df, target)
excluded_cols = set(exclusion_table['column'])
feature_columns = [col for col in df.columns if col != target and col not in excluded_cols]

model_df = df[[target] + feature_columns].copy()
y = pd.to_numeric(model_df[target], errors='coerce')
valid_target_mask = y.notna()
model_df = model_df.loc[valid_target_mask].reset_index(drop=True)
y = y.loc[valid_target_mask].reset_index(drop=True)
X = model_df[feature_columns].copy()

display(Markdown(f'**Selected target:** `{target}`'))
display(Markdown(f'**Modeling rows with non-missing target:** {len(model_df):,}'))
display(Markdown(f'**Predictor count after exclusions:** {X.shape[1]}'))

display(exclusion_table.head(30))
display(leakage_audit.head(20))

exclusion_table.to_csv(OUTDIR / 'excluded_features.csv', index=False)
leakage_audit.to_csv(OUTDIR / 'target_leakage_audit.csv', index=False)
save_json(
    {
        'dataset_path': str(DATA_PATH),
        'target': target,
        'rows_raw': int(len(df_raw)),
        'rows_modeled': int(len(model_df)),
        'predictor_count': int(X.shape[1]),
    },
    OUTDIR / 'data_summary.json',
)


In [ ]:
groups, group_cols = build_groups(model_df)
cv_splitter, holdout_splitter, validation_note = choose_validation_strategy(groups, group_cols)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(y, bins=40, kde=True, ax=axes[0], color='steelblue')
axes[0].set_title(f'Target distribution: {target}')
axes[0].set_xlabel(target)

sns.boxplot(x=y, ax=axes[1], color='lightgreen')
axes[1].set_title('Target spread')
axes[1].set_xlabel(target)

fig.tight_layout()
fig.savefig(FIGDIR / 'target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

missing_summary = (
    X.isna().mean().sort_values(ascending=False).rename('missing_rate').reset_index().rename(columns={'index': 'feature'})
)
display(Markdown(validation_note))
display(missing_summary.head(20))


## Baselines and model comparison

All imputing and categorical handling are fit **inside the pipeline**, so each fold learns preprocessing only from its training subset. High-cardinality categoricals are handled with `OneHotEncoder(min_frequency=20)` when the installed scikit-learn version supports it; otherwise the encoder falls back to `handle_unknown='ignore'`.


In [ ]:
preprocessor, numeric_features, categorical_features = build_preprocessor(X)

models = {
    'Mean baseline': DummyRegressor(strategy='mean'),
    'Random forest': RandomForestRegressor(
        n_estimators=400,
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS,
    ),
}

if LGBMRegressor is not None:
    models['LightGBM'] = LGBMRegressor(
        n_estimators=500,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS,
    )

if XGBRegressor is not None:
    models['XGBoost'] = XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_lambda=1.0,
        objective='reg:squarederror',
        tree_method='hist',
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS,
        verbosity=0,
    )

scoring = {
    'mae': 'neg_mean_absolute_error',
    'rmse': make_scorer(rmse_score, greater_is_better=False),
    'r2': 'r2',
}

results = []
uses_grouped_validation = groups is not None and validation_note.startswith('Grouped validation')
for model_name, estimator in models.items():
    pipeline = Pipeline([
        ('preprocess', clone(preprocessor)),
        ('model', estimator),
    ])
    cv_kwargs = {'cv': cv_splitter, 'scoring': scoring, 'n_jobs': 1, 'error_score': 'raise'}
    if uses_grouped_validation:
        scores = cross_validate(pipeline, X, y, groups=groups, **cv_kwargs)
    else:
        scores = cross_validate(pipeline, X, y, **cv_kwargs)

    results.append(
        {
            'model': model_name,
            'cv_mae_mean': -scores['test_mae'].mean(),
            'cv_mae_std': scores['test_mae'].std(),
            'cv_rmse_mean': -scores['test_rmse'].mean(),
            'cv_rmse_std': scores['test_rmse'].std(),
            'cv_r2_mean': scores['test_r2'].mean(),
            'cv_r2_std': scores['test_r2'].std(),
        }
    )

results_df = pd.DataFrame(results).sort_values(['cv_rmse_mean', 'cv_mae_mean']).reset_index(drop=True)
display(results_df)
results_df.to_csv(OUTDIR / 'model_comparison.csv', index=False)


In [ ]:
if groups is not None and validation_note.startswith('Grouped validation'):
    train_idx, test_idx = next(holdout_splitter.split(X, y, groups))
else:
    train_idx, test_idx = next(holdout_splitter.split(X, y))

X_train = X.iloc[train_idx].reset_index(drop=True)
X_test = X.iloc[test_idx].reset_index(drop=True)
y_train = y.iloc[train_idx].reset_index(drop=True)
y_test = y.iloc[test_idx].reset_index(drop=True)
analysis_test = model_df.iloc[test_idx].reset_index(drop=True)

best_model_name = results_df.iloc[0]['model']
best_estimator = clone(models[best_model_name])
train_preprocessor, train_num_features, train_cat_features = build_preprocessor(X_train)
best_pipeline = Pipeline([
    ('preprocess', train_preprocessor),
    ('model', best_estimator),
])

best_pipeline.fit(X_train, y_train)
y_pred = best_pipeline.predict(X_test)

holdout_metrics = {
    'best_model': best_model_name,
    'validation_strategy': validation_note,
    'holdout_mae': float(mean_absolute_error(y_test, y_pred)),
    'holdout_rmse': float(np.sqrt(mean_squared_error(y_test, y_pred))),
    'holdout_r2': float(r2_score(y_test, y_pred)),
    'train_rows': int(len(X_train)),
    'test_rows': int(len(X_test)),
}
display(pd.DataFrame([holdout_metrics]))

context_columns = [col for col in ['make', 'model', 'year', 'class', 'fuel_type', 'fuel_type1', 'drive'] if col in analysis_test.columns]
holdout_export = analysis_test[context_columns].copy()
holdout_export['y_true'] = y_test
holdout_export['y_pred'] = y_pred
holdout_export['residual'] = holdout_export['y_true'] - holdout_export['y_pred']
holdout_export['abs_error'] = holdout_export['residual'].abs()

holdout_export.to_csv(OUTDIR / 'holdout_predictions.csv', index=False)
save_json(holdout_metrics, OUTDIR / 'holdout_metrics.json')
joblib.dump(best_pipeline, OUTDIR / 'best_model.joblib')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.scatterplot(data=holdout_export, x='y_true', y='y_pred', alpha=0.45, ax=axes[0])
limits = [
    min(holdout_export['y_true'].min(), holdout_export['y_pred'].min()),
    max(holdout_export['y_true'].max(), holdout_export['y_pred'].max()),
]
axes[0].plot(limits, limits, linestyle='--', color='black')
axes[0].set_title(f'{best_model_name}: predicted vs. actual')
axes[0].set_xlabel('Actual MPG')
axes[0].set_ylabel('Predicted MPG')

sns.scatterplot(data=holdout_export, x='y_pred', y='residual', alpha=0.45, ax=axes[1])
axes[1].axhline(0, linestyle='--', color='black')
axes[1].set_title('Residuals by prediction')
axes[1].set_xlabel('Predicted MPG')
axes[1].set_ylabel('Residual (actual - predicted)')

fig.tight_layout()
fig.savefig(FIGDIR / 'holdout_diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
perm_sample_n = min(2000, len(X_test))
perm_X = X_test.sample(n=perm_sample_n, random_state=RANDOM_STATE)
perm_y = y_test.loc[perm_X.index]

permutation = permutation_importance(
    best_pipeline,
    perm_X,
    perm_y,
    scoring='neg_mean_absolute_error',
    n_repeats=10,
    random_state=RANDOM_STATE,
    n_jobs=1,
)

importance_df = pd.DataFrame(
    {
        'feature': X.columns,
        'importance_mean': permutation.importances_mean,
        'importance_std': permutation.importances_std,
    }
).sort_values('importance_mean', ascending=False)

display(importance_df.head(20))
importance_df.to_csv(OUTDIR / 'permutation_importance.csv', index=False)

fig, ax = plt.subplots(figsize=(10, 6))
plot_df = importance_df.head(15).sort_values('importance_mean')
ax.barh(plot_df['feature'], plot_df['importance_mean'], xerr=plot_df['importance_std'], color='steelblue')
ax.set_title('Permutation importance (top 15)')
ax.set_xlabel('Increase in MAE when shuffled')
fig.tight_layout()
fig.savefig(FIGDIR / 'permutation_importance.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
if DO_SHAP:
    if shap is None:
        print('SHAP is not installed; using permutation importance as the fallback explanation.')
    else:
        try:
            model = best_pipeline.named_steps['model']
            preprocessor = best_pipeline.named_steps['preprocess']
            shap_sample = X_test.sample(n=min(SHAP_SAMPLE_N, len(X_test)), random_state=RANDOM_STATE)
            transformed = preprocessor.transform(shap_sample)
            if hasattr(transformed, 'toarray'):
                transformed = transformed.toarray()
            feature_names = preprocessor.get_feature_names_out()
            explainer = shap.Explainer(model, transformed, feature_names=feature_names)
            explanation = explainer(transformed)
            shap.summary_plot(explanation, features=transformed, feature_names=feature_names, show=True)
        except Exception as exc:
            print(f'SHAP was requested but could not be computed: {exc}')
            print('Permutation importance remains available in artifacts/permutation_importance.csv.')
else:
    print('DO_SHAP is False; permutation importance is the default lightweight explanation.')


In [ ]:
worst_errors = holdout_export.sort_values('abs_error', ascending=False).head(15)
display(worst_errors)
worst_errors.to_csv(OUTDIR / 'worst_errors.csv', index=False)

group_summaries = []
for group_col in ['class', 'fuel_type', 'fuel_type1', 'make']:
    if group_col in holdout_export.columns:
        group_summary = (
            holdout_export.groupby(group_col)
            .agg(
                n=('abs_error', 'size'),
                mae=('abs_error', 'mean'),
                rmse=('residual', lambda s: np.sqrt(mean_squared_error(np.zeros(len(s)), s))),
            )
            .query('n >= 5')
            .sort_values('mae', ascending=False)
            .head(10)
            .reset_index()
        )
        group_summary.insert(0, 'grouping_column', group_col)
        group_summaries.append(group_summary)

if group_summaries:
    group_error_summary = pd.concat(group_summaries, ignore_index=True)
    display(group_error_summary)
    group_error_summary.to_csv(OUTDIR / 'group_error_summary.csv', index=False)
else:
    group_error_summary = pd.DataFrame()
    print('No categorical grouping columns were available for group-level error analysis.')

powertrain_col = first_present(holdout_export, ['fuel_type', 'fuel_type1'])
if powertrain_col is not None:
    powertrain_summary = (
        holdout_export.assign(
            is_hybrid_or_ev=holdout_export[powertrain_col].fillna('').astype(str).str.contains('electric|hybrid', case=False)
        )
        .groupby('is_hybrid_or_ev')
        .agg(
            n=('abs_error', 'size'),
            mae=('abs_error', 'mean'),
            rmse=('residual', lambda s: np.sqrt(mean_squared_error(np.zeros(len(s)), s))),
        )
    )
    display(powertrain_summary)
else:
    powertrain_summary = pd.DataFrame()

run_summary = {
    'target': target,
    'validation_strategy': validation_note,
    'group_columns': group_cols,
    'numeric_feature_count': len(numeric_features),
    'categorical_feature_count': len(categorical_features),
    'best_model': best_model_name,
    'cv_results': results_df.to_dict(orient='records'),
    'holdout_metrics': holdout_metrics,
}
save_json(run_summary, OUTDIR / 'run_summary.json')
print('Saved modeling artifacts to', OUTDIR.resolve())


## Interpretation and limitations

Use the saved tables and figures as the basis for the portfolio story in the README:

- **What usually works well:** repeated vehicle families with stable gasoline drivetrains and well-represented classes.
- **Where errors often grow:** hybrids/EVs, sparse trims, unusual engine/transmission combinations, and vehicles from years that differ meaningfully from the dominant training distribution.
- **Why grouped validation matters:** random row splits can place nearly identical trims in both train and test, inflating performance.
- **What this model does *not* prove:** perfect generalization to future model years or to newly introduced powertrains. If forecasting future releases is the goal, add a strict year-based holdout as a follow-up analysis.

The notebook saves compact artifacts under `artifacts/`:
- `model_comparison.csv`
- `holdout_metrics.json`
- `holdout_predictions.csv`
- `permutation_importance.csv`
- `worst_errors.csv`
- `group_error_summary.csv`
- `figures/target_distribution.png`
- `figures/holdout_diagnostics.png`
- `figures/permutation_importance.png`
